# Colocalization plots

cell2location (q05) composition of the ECM neighborhoods, TIL zones and pathology annotations
(moved from `analysis/cell2loc/colocalization_plots.ipynb`).

Produces:
- **Supp Table 3b** (`4b_ECM_neighborhood_mean_gene_expression`): mean expression of every gene
  per ECM neighborhood; **Supp Table 3c** (`4c_*`): mean cell-type composition per ECM
  neighborhood, TIL zone and pathology annotation at the four curated levels. (The text cites
  them as Supp Table 4b–c / 3c; the EFS files are named 4b, 4c.)
- **Fig 4e**: cell-type marker dotplot of the ECM neighborhoods (level 4), with the same dotplot
  for the other levels and annotations.
- **Supp Fig 4e**: ECM neighborhood fractions per sample (stacked bars with the centre bar), with
  the same plot for TIL zones and pathology annotations.

Fig 4f (ECM neighborhoods × pathology annotations) is the standardized-residual heatmap of
`14_pathologist_annotation_analyses.py`, not the dotplot of the former cell 22: the published panel
is that heatmap (its data equal `results/hne/ecm_pathology_figures/Fig_residual_heatmap.csv`).
Supp Fig 4e is this notebook's bar plot (with the centre annotation), not the per-patient bar plot of
`11_ECM_niches_v4` (former cell 28).

Changes from the original notebook:
- Paths from `gbmcollaboration.paper.config`; figures and tables go to `OUT` with `output.save` /
  `output.save_table` (paper panels also to the panels folder) instead of EFS `figures/visium/` and
  `supplementary_tables/`; the published versions are read back and compared.
- Loading (cell 7): `neighborhoods.load_spots()` gives the q05 abundances of
  `load_spatial_adata` without the expression matrix; cell 9's gene means per neighborhood come
  from `neighborhoods.pseudobulk_sums` (same means; the full dense matrix is > 100 GB).
- Fig 4e (cell 16): run with the defaults of `dotplot_composite_score` (the published version;
  the cell last ran with `center_zero=True, cmap='bwr'`).
- Bug fixed: `col_prefix` kept the "ECM_" of the previous annotation for `pathology_annotations`
  (cells 12–24); now empty for every annotation but the ECM neighborhoods.
- Removed: cell 2 (FileLink), 8 (duplicate of 7), 12 (clustermap dotplots), 14 (earlier version
  of 16), 18 (`plot_neighborhood_marker_dotplot_from_agg` is not defined: NameError), 20 and 22
  (pathology × neighborhood dotplots, not paper panels; see Fig 4f above), unused imports.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from gbmcollaboration.paper import config, neighborhoods, output
from gbmcollaboration.utils import pathology_annotation_loader as patho
from gbmcollaboration.viz.cell2loc_plotting import dotplot_composite_score
from gbmcollaboration.viz.stacked_barplot import clustered_stacked_barplot

# Params

In [ ]:
OUT = output.setup("13_colocalization_plots")
path_ecm_annot = config.ECM_ANNOTATIONS_CSV
path_til_annot = config.TIL_ZONES_CSV
path_cell_type_info = config.GLIMMER_MARKERS_XLSX
FIGURES_DIR_CUSTOM = "custom_colocalization"          # subfolders of OUT
FIGURES_DIR_BARPLOTS = "stacked_patient_barplots"

# Outputs of the last run, to compare with
PUBLISHED_TABLES = config.EFS / "supplementary_tables"
PUBLISHED_FIGURES = config.EFS / "figures/visium"

# Load annotations

In [ ]:
# load ECM and TIL annotations
df_ecm_annot = pd.read_csv(path_ecm_annot,index_col='spot_id')
df_til_annot = pd.read_csv(path_til_annot,index_col='spot_id')

# load pathology annotations
cfg = patho.DataConfig(
    s3_bucket="pa-bi4jtrdsa151p4d3m-jepjxh9qrwgxphe8hd3r6exicahnoeuw1b-s3alias",
    s3_prefixes=(
        "fd319842-9157-4f04-bd4f-7a36ce6a0ba7/Non_mosaic_upload/Mosaic Annotations/",  # CHUV
        "2ca49430-8b5c-4504-ad96-344019c90e84/",  # UKER
    ),
)

acfg = patho.AnalysisConfig(
    min_spots_per_annotation=0,
    keep_only_shared_annotations=False,
)

df_patho_annot = patho.load_and_prepare(cfg, acfg)
df_all_annot = df_ecm_annot.join([df_til_annot[['TIL_positive','TIL_zone']], df_patho_annot['pathology_annotations']])
df_all_annot[['ECM_leiden','ECM_neighborhood']]=df_all_annot[['ECM_leiden','ECM_neighborhood']].astype(int)

# Load data

cell2location q05 abundances (`deconv_cohort_level_4`) of every spot, with the loader exclusions
of `load_spatial_adata` (`neighborhoods.load_spots`; the expression matrix is not loaded here).

In [ ]:
spots, df_q05 = neighborhoods.load_spots()
df_q05 = df_q05.fillna(0.)

# map cell types names to paper names and create simplified deconv matrix
df_markers, df_ct_map = pd.read_excel(path_cell_type_info,sheet_name=None).values()
assert all(df_q05.columns.isin(df_ct_map['celltype_level4_scanvi']))

# assert index matches with metadata
assert all(df_q05.index == df_all_annot.index)

# normalize
df_q05_norm = df_q05.div(df_q05.sum(axis=1), axis=0)

# join with metadata for each level and create simplified deconv matrix
# also get average per annotation
annot_keys = [
    'TIL_zone',
    'ECM_neighborhood',
    'pathology_annotations'
]

dfs_q05 = {}
dfs_q05_norm = {}
dfs_q05_sum_per_annot = {}
dfs_q05_norm_mean_per_annot = {}
for i in tqdm(range(1,5)):
    lvl = f'celltype_level{i}_curated'
    _map = df_ct_map.set_index('celltype_level4_scanvi')[lvl]

    dfs_q05[lvl] = df_q05.T.groupby(_map).sum().T.join(df_all_annot)
    dfs_q05_norm[lvl] = df_q05_norm.T.groupby(_map).sum().T.join(df_all_annot)

    ct_cols = dfs_q05_norm[lvl].columns.difference(df_all_annot.columns)

    dfs_q05_sum_per_annot[lvl] = {}
    dfs_q05_norm_mean_per_annot[lvl] = {}
    for k in annot_keys:
        dfs_q05_sum_per_annot[lvl][k] = dfs_q05[lvl].groupby(k)[ct_cols].sum(0).T
        dfs_q05_norm_mean_per_annot[lvl][k] = dfs_q05_norm[lvl].groupby(k)[ct_cols].mean(0).T

# Supp Tables 3b, 3c

3b: mean log-normalised expression of every gene per ECM neighborhood (over spots), from per
section × neighborhood sums (`neighborhoods.pseudobulk_sums`) instead of the full matrix.
3c: mean cell-type composition per ECM neighborhood, TIL zone and pathology annotation.

In [ ]:
pb_sums, pb_n = neighborhoods.pseudobulk_sums(df_all_annot['ECM_neighborhood'])
df_gex_ecm_mean = pb_sums.groupby(level='label').sum().div(pb_n.groupby(level='label').sum(), axis=0)
df_gex_ecm_mean.index = df_gex_ecm_mean.index.astype(int).rename('ECM_neighborhood')
df_gex_ecm_mean = df_gex_ecm_mean.sort_index()
output.save_table(OUT, '4b_ECM_neighborhood_mean_gene_expression', df_gex_ecm_mean, panel='Supp Table 3b')

for ct_lvl in dfs_q05_norm_mean_per_annot.keys():
    for k in dfs_q05_norm_mean_per_annot[ct_lvl].keys():
        output.save_table(OUT, f'4c_{k}_{ct_lvl}_mean_cell_type_composition', dfs_q05_norm_mean_per_annot[ct_lvl][k],
                          panel=f'Supp Table 3c {k} {ct_lvl}')

In [ ]:
# Compare with the published tables
pub = pd.read_csv(PUBLISHED_TABLES / '4b_ECM_neighborhood_mean_gene_expression.csv', index_col=0)
pub.index = pub.index.astype(int)   # written with float labels (0.0, 1.0, ...)
print(f"3b: published {pub.shape}, here {df_gex_ecm_mean.shape}; same genes: {set(pub.columns) == set(df_gex_ecm_mean.columns)}; "
      f"max |difference| = {np.abs(pub.to_numpy() - df_gex_ecm_mean.loc[pub.index, pub.columns].to_numpy()).max():.2e}")
for ct_lvl in dfs_q05_norm_mean_per_annot:
    for k in dfs_q05_norm_mean_per_annot[ct_lvl]:
        pub = pd.read_csv(PUBLISHED_TABLES / f'4c_{k}_{ct_lvl}_mean_cell_type_composition.csv', index_col=0)
        new = dfs_q05_norm_mean_per_annot[ct_lvl][k].rename(columns=str)
        pub.columns = pub.columns.astype(str).str.replace(r'\.0$', '', regex=True)   # float labels then
        d = np.abs(pub.to_numpy() - new.loc[pub.index, pub.columns].to_numpy()).max()
        print(f"3c {k:<22} {ct_lvl}: published {pub.shape}, here {new.shape}, max |difference| {d:.2e}")

# Custom dotplots (Fig 4e)

The published Fig 4e is the ECM neighborhood × level 4 dotplot with the function defaults
(marker score composition × log2 FC⁺ on RdPu from 0; `figures/visium/custom_colocalization/`),
not the `center_zero=True, cmap='bwr'` version the cell last ran with.

In [ ]:
plot_annots = {
    'TIL_zone': None,
    'ECM_neighborhood': None,
    'pathology_annotations': None
}

for i in tqdm(range(1,5)):
    lvl = f'celltype_level{i}_curated'

    for k, cat_subset in plot_annots.items():

        if k == 'ECM_neighborhood':
            col_prefix = 'ECM_'
        else:
            col_prefix = ''

        df_q05_sum = dfs_q05_sum_per_annot[lvl][k]

        # upper = np.quantile(df_q05_sum,0.99)
        df_plot = df_q05_sum.copy()
        # df_plot = df_plot.clip(upper=upper)
        df_plot.columns = col_prefix+df_plot.columns.astype(str)

        if cat_subset is not None:
            df_plot = df_plot.loc[cat_subset]

        fig, ax, metrics = dotplot_composite_score(
            abund_per_nh=df_plot.T, 
            top_n_per_niche=5, 
            show_excluded_rows=False,
        )

        # save plot
        panel = 'Fig 4e' if (lvl, k) == ('celltype_level4_curated', 'ECM_neighborhood') else None
        output.save(OUT, f'{FIGURES_DIR_CUSTOM}/dotplot_{lvl}_{k}', fig, df_plot, panel=panel)
        if panel:
            fig4e_rows = [t.get_text() for t in ax.get_yticklabels()]
            fig4e_cols = [t.get_text() for t in ax.get_xticklabels()]
        plt.close(fig)

In [ ]:
# Compare Fig 4e with the published version (figures/visium/custom_colocalization/)
pub = pd.read_csv(PUBLISHED_FIGURES / 'custom_colocalization/dotplot_celltype_level4_curated_ECM_neighborhood.csv', index_col=0)
pub.columns = pub.columns.str.replace(r'\.0$', '', regex=True)   # ECM_3.0 -> ECM_3
new = pd.read_csv(OUT / f'{FIGURES_DIR_CUSTOM}/dotplot_celltype_level4_curated_ECM_neighborhood.csv', index_col=0)
print(f"plotted abundances: max |difference| = {np.abs(pub.to_numpy() - new.loc[pub.index, pub.columns].to_numpy()).max():.2e}")
print("rows:", fig4e_rows)
print("columns:", fig4e_cols)

# Patient composition plots (Supp Fig 4e)

In [ ]:
plot_annots = {
    'TIL_zone': None,
    'ECM_neighborhood': None,
    'pathology_annotations': None
}

for k, cat_subset in plot_annots.items():

    if k == 'ECM_neighborhood':
        col_prefix = 'ECM_'
    else:
        col_prefix = ''

    df_patient_props = pd.crosstab(df_all_annot['sample'],df_all_annot[k])
    df_patient_props = df_patient_props.div(df_patient_props.sum(axis=1),axis=0)
    
    if k == 'TIL_zone':
        cat_order = df_patient_props.sum(0).sort_values(ascending=False).index.tolist()
        df_patient_props = df_patient_props.sort_values(cat_order)
        cluster_cols=False
        cluster_rows=False
    else:
        cluster_cols=True
        cluster_rows=True


    col_annot = df_patient_props.index.str.split('_').str[0]

    fig, ax, ordered = clustered_stacked_barplot(
        df_patient_props,
        cluster_cols=cluster_cols,
        cluster_rows=cluster_rows,
        col_annotation=col_annot,
        col_annotation_cmap="Set2",
        col_annotation_label="Center",
        figsize=(25,8),
        legend_kwargs = dict(
            fontsize=20,
            title_fontsize=20,
        )
    )
    # save plot
    output.save(OUT, f'{FIGURES_DIR_BARPLOTS}/barplot_{k}', fig, df_patient_props,
                panel='Supp Fig 4e' if k == 'ECM_neighborhood' else None)
    plt.close(fig)

In [ ]:
# Compare with the published bar plot data (figures/visium/stacked_patient_barplots/)
pub = pd.read_csv(PUBLISHED_FIGURES / 'stacked_patient_barplots/barplot_ECM_neighborhood.csv', index_col=0)
new = pd.read_csv(OUT / f'{FIGURES_DIR_BARPLOTS}/barplot_ECM_neighborhood.csv', index_col=0)
pub.columns = pub.columns.str.replace(r'\.0$', '', regex=True)   # float labels then
print(f"Supp Fig 4e: {new.shape} (published {pub.shape}); max |difference| = "
      f"{np.abs(pub.to_numpy() - new.loc[pub.index, pub.columns].to_numpy()).max():.2e}")

In [ ]:
output.versions(["anndata", "scanpy", "statsmodels"])